# Разведочный анализ FPB + MyFoodRepo v2.1

Анализ готовой YOLO-сборки из ноутбука 02: примеры изображений, размеры изображений и объектов, точные и похожие копии, кандидаты в выбросы и распределения классов. **Удаление похожих кадров по заданным порогам включается флагом `REMOVE_SIMILAR_IMAGES`.** Перед применением показывается план очистки. Редкие размеры, низкая резкость и другие статистические выбросы автоматически не удаляются.

В `processed/` нет масок сегментации: COCO-полигоны MyFoodRepo уже преобразованы в рамки. Поэтому анализ «размеров масок» заменён анализом ширины, высоты и площади **bbox**, без создания искусственных масок. У FPB test нет открытых меток: его классы неизвестны, их нельзя показывать как нулевые частоты.

In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
from collections import Counter

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/dataset_pipeline.py").is_file())
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
os.environ.setdefault("MPLCONFIGDIR", str(DATA / "cache/matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scripts import dataset_pipeline as dp
from scripts import dataset_eda as eda
from scripts import dataset_split as ds


In [ ]:
MODE = "full"                   # "full" — итоговая сборка; "preview" — проверочный сэмпл.
SEED = 42 
WORKERS = 4                     # Число потоков чтения изображений.
RECOMPUTE_STATS = False          # True — перечитать фотографии, обновить кеш характеристик.
MAX_DHASH_DISTANCE = 3           # Допустимое число различающихся битов dHash: 0–3.
MAX_RGB_MAE = 0.004               # Средняя разница RGB-миниатюр 32×32 в шкале 0–1.
ASPECT_TOLERANCE = 0.03          # Предел abs(log(ratio_a / ratio_b)) для похожих кадров.
REMOVE_SIMILAR_IMAGES = True   # True — удалять похожие кадры по трём порогам выше, включая точные копии.

RESPLIT_DATASET = True           # Применить новое разбиение размеченной части после EDA.
SPLIT_FRACTIONS = (0.80, 0.15, 0.05)  # Доли фотографий: train, valid, test; test без меток исключён.

OUTPUT = dp.dataset_path(DATA, MODE)
records, images, objects, names = eda.load_export(OUTPUT)
MANIFEST_HASH = hashlib.sha256((OUTPUT / "manifest.jsonl").read_bytes()).hexdigest()
print("Сборка:", OUTPUT.relative_to(ROOT))
print("Сборка:", OUTPUT.relative_to(ROOT))
print(f"Изображений: {len(images):,}; размеченных объектов: {len(objects):,}; классов: {len(names)}")
display(images.groupby(["source", "split"]).agg(images=("image", "size"), labeled_images=("labeled", "sum"), objects=("n_objects", lambda values: values.sum(min_count=1))))

## 1. Просмотр небольшой выборки

Выбираем до двух примеров из каждой пары «источник — выборка». Зелёные рамки и подписи соответствуют итоговым классам. Для неразмеченного test показывается само изображение.

In [ ]:
sample = images.groupby(["source", "split"], group_keys=False).sample(frac=1, random_state=SEED).groupby(["source", "split"]).head(2)
fig = dp.show_records([records[i] for i in sample.record_i], names, output_root=OUTPUT, count=12, title="Примеры готового датасета")
if fig is not None: plt.close(fig)
display(sample[["source", "source_id", "split", "width", "height", "n_objects"]])

## 2. Размеры изображений и рамок

Читаем фактические размеры, цветовой режим и характеристики качества. Для JPEG используются уменьшенное декодирование и миниатюры; резкость и яркость — сравнительные признаки, а не экспертная оценка. Проверяется читаемость файла. Кеш связан с SHA-256 manifest; после ручного изменения фотографий включите `RECOMPUTE_STATS`.

Полный RGB-хеш каждого изображения уже записан при сборке в ноутбуке 02. Здесь он используется для поиска точных копий; перед удалением найденные файлы повторно читаются целиком. Это не повторное хеширование всех фотографий.

In [ ]:
if "thumbnails" in globals(): del thumbnails  # Освободить mmap перед обновлением кеша в Windows.
features, thumbnails = eda.scan_images(DATA, OUTPUT, images, workers=WORKERS, overwrite=RECOMPUTE_STATS)
valid_features = features[features.error == ""].copy()
valid_features["aspect"] = valid_features.width / valid_features.height
valid_features["megapixels"] = valid_features.width * valid_features.height / 1e6
print("Ошибок чтения:", int((features.error != "").sum()))
display(valid_features.groupby("source")[["width", "height", "aspect", "megapixels"]].agg(["min", "median", "max"]))
display(valid_features.groupby(["width", "height"]).size().sort_values(ascending=False).head(15).rename("images").to_frame())
display(valid_features.groupby("mode").size().rename("images").to_frame())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for source, group in valid_features.groupby("source"):
    axes[0, 0].hist(group.width, bins=50, histtype="step", label=source)
    axes[0, 1].hist(group.height, bins=50, histtype="step", label=source)
    axes[1, 0].hist(group.aspect, bins=60, histtype="step", label=source)
    axes[1, 1].scatter(group.width, group.height, s=5, alpha=0.15, label=source)
for axis, title in zip(axes.flat, ["Ширина изображения, px", "Высота изображения, px", "Отношение width / height", "Размеры изображений, px"]):
    axis.set_title(title); axis.legend(); axis.grid(alpha=0.2)
axes[1, 1].set_xlabel("width"); axes[1, 1].set_ylabel("height")
fig.tight_layout(); plt.show(); plt.close(fig)

boxes = objects.copy()
actual_sizes = features.set_index("record_i")
boxes["width_px"] = boxes.width_norm * boxes.record_i.map(actual_sizes.width)
boxes["height_px"] = boxes.height_norm * boxes.record_i.map(actual_sizes.height)
display(boxes.groupby("source")[["width_px", "height_px", "area_norm"]].describe())
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for source, group in boxes.groupby("source"):
    axes[0].hist(np.log10(group.area_norm.clip(lower=1e-12)), bins=50, histtype="step", label=source)
    sampled = group.sample(min(10000, len(group)), random_state=SEED)
    axes[1].scatter(sampled.width_px, sampled.height_px, s=4, alpha=0.15, label=source)
axes[0].set_title("log10 относительной площади bbox"); axes[0].legend()
axes[1].set(xscale="log", yscale="log", xlabel="bbox width, px", ylabel="bbox height, px", title="Размеры рамок")
axes[2].hist(images.loc[images.labeled, "n_objects"], bins=30)
axes[2].set_title("Объектов на размеченном фото")
fig.tight_layout(); plt.show(); plt.close(fig)

## 3. Точные дубликаты и очень похожие изображения

Точные копии группируются по пиксельному SHA-256 из manifest (в хеш включены размеры). Для **кандидатов в похожие** используется независимый dHash миниатюры, близкое соотношение сторон и средняя абсолютная разница RGB. Четыре 16-битные части dHash позволяют искать пары с расстоянием до трёх бит без полного попарного сравнения всех фотографий.

Похожие кадры могут показывать разные ракурсы и порции. Маленькая разница миниатюр или одинаковые классы не доказывают дубликат: эти кандидаты удаляются только при включённом `REMOVE_SIMILAR_IMAGES` после построения плана очистки. Поиск не является исчерпывающим семантическим сравнением: он может пропускать кадры с другим кадрированием, освещением или большим поворотом.

In [ ]:
current_manifest_hash = hashlib.sha256((OUTPUT / "manifest.jsonl").read_bytes()).hexdigest()
if current_manifest_hash != MANIFEST_HASH or images.key.tolist() != features.key.tolist():
    raise RuntimeError("Сборка изменилась: запустите ноутбук с начала, включая чтение характеристик.")

exact_groups = images.groupby("pixel_sha256").filter(lambda group: len(group) > 1)
print("Групп точных копий по manifest:", exact_groups.pixel_sha256.nunique())
print("Избыточных точных копий:", len(exact_groups) - exact_groups.pixel_sha256.nunique())

pair_params = {"manifest": MANIFEST_HASH, "hamming": MAX_DHASH_DISTANCE, "mae": MAX_RGB_MAE,
               "aspect": ASPECT_TOLERANCE, "method": 1}
pair_cache = DATA / "metadata" / (OUTPUT.name + ".eda_pairs.json")
cached = dp.read_json(pair_cache) if pair_cache.exists() and not RECOMPUTE_STATS else {}
if cached.get("parameters") == pair_params:
    pairs = pd.DataFrame(cached["pairs"], columns=["a", "b", "hamming", "rgb_mae"])
    print("Использован кеш похожих пар.")
else:
    pairs = eda.similar_pairs(features, thumbnails, hamming=MAX_DHASH_DISTANCE,
                            mae=MAX_RGB_MAE, aspect_tolerance=ASPECT_TOLERANCE)
    dp.dump_json(pair_cache, {"parameters": pair_params, "pairs": pairs.to_dict("records")})
if len(pairs):
    by_i = images.set_index("record_i")
    pairs["exact_hash"] = pairs.a.map(by_i.pixel_sha256) == pairs.b.map(by_i.pixel_sha256)
    pairs["cross_split"] = pairs.a.map(by_i.split) != pairs.b.map(by_i.split)
    def class_set(i):
        return None if records[i]["objects"] is None else tuple(sorted(o["class_id"] for o in records[i]["objects"]))
    pairs["same_classes"] = [class_set(a) == class_set(b) if class_set(a) is not None and class_set(b) is not None else None
                             for a, b in zip(pairs.a, pairs.b)]
    print("Кандидатов в похожие пары:", int((~pairs.exact_hash).sum()), "; между выборками:", int(pairs.cross_split.sum()))
    display(pairs.sort_values(["rgb_mae", "hamming"]).head(20))
else:
    print("Похожих пар по выбранным порогам не найдено.")


In [ ]:
if len(pairs):
    for number, pair in enumerate(pairs.sort_values(["rgb_mae", "hamming"]).head(10).itertuples(), start=1):
        print(f"Пара {number}: dHash={pair.hamming}, RGB MAE={pair.rgb_mae:.5f}, разные выборки={pair.cross_split}")
        selected = [records[pair.a], records[pair.b]]
        display(pd.DataFrame([{k: row[k] for k in ["source", "source_id", "split", "image"]} for row in selected]))
        fig = dp.show_records(selected, names, output_root=OUTPUT, count=2, title=f"Похожая пара {number} — до применения очистки")
        if fig is not None: plt.close(fig)

## 4. Удаление точных и похожих изображений

`REMOVE_SIMILAR_IMAGES = True` включает очистку по всем трём условиям одновременно: `MAX_DHASH_DISTANCE`, `MAX_RGB_MAE` и `ASPECT_TOLERANCE`. Точные пиксельные копии также включаются в этот режим.

Перед удалением показано число исключаемых изображений и таблица **сохраняемая запись → удаляемая запись**. Сначала сохраняется запись **с разметкой**: при совпадении с неразмеченным test удаляется запись без меток. Среди размеченных записей приоритет: `test` → `val` → `train`, затем источник и исходный ID. Для двух неразмеченных записей используется приоритет выборки и исходный ID. Каждая удаляемая запись должна напрямую соответствовать порогам относительно остающейся записи. Сходство A–B и B–C само по себе не позволяет удалить C при сохранении A, если A–C не соответствует порогам.

В режиме похожих изображений отбор определяется картинками: различия классов или рамок не блокируют удаление; признак `same_classes` вынесен в план. При выборе только точных копий сохраняется прежнее поведение: конфликтующие аннотации остаются для отдельной проверки. Статистические выбросы автоматически не удаляются.

Перед изменением файлов выбранные пары повторно проверяются по изображениям. Очистка обновляет `images/`, `labels/`, manifest, списки выборок, аудит `excluded_records.json` и контрольную сумму сборки. Для похожих кадров в аудите записываются причина `similar_image`, сохраняемый ID, расстояния и пороги; для точных — `pixel_duplicate`. Исходники в `data/raw/` сохраняются. При ошибке проверки изменения откатываются.

Если пороги или manifest изменились после поиска пар, очистка останавливается: нужно повторить поиск, а после изменения состава сборки — запустить ноутбук с начала. Кеш поиска связан с manifest и порогами. Итоговые распределения ниже рассчитываются уже после очистки.

In [ ]:
expected_params = {"manifest": hashlib.sha256((OUTPUT / "manifest.jsonl").read_bytes()).hexdigest(),
                   "hamming": MAX_DHASH_DISTANCE, "mae": MAX_RGB_MAE, "aspect": ASPECT_TOLERANCE, "method": 1}
if pair_params != expected_params:
    raise RuntimeError("Пороги или manifest изменились. Повторите поиск похожих пар перед очисткой.")
cleanup_plan = eda.similar_removal_plan(records, pairs, hamming=MAX_DHASH_DISTANCE,
                                      mae=MAX_RGB_MAE, aspect_tolerance=ASPECT_TOLERANCE)
print("План при включении REMOVE_SIMILAR_IMAGES: удалить", len(cleanup_plan), "изображений")
if len(cleanup_plan):
    display(cleanup_plan.groupby(["reason", "remove_split"]).size().rename("images_to_remove").to_frame())
    display(cleanup_plan.head(20))
    print("Пары с различающимися наборами классов:", int((cleanup_plan.same_classes == False).sum()))


In [ ]:
if REMOVE_SIMILAR_IMAGES:
    if pair_params != {"manifest": hashlib.sha256((OUTPUT / "manifest.jsonl").read_bytes()).hexdigest(),
                       "hamming": MAX_DHASH_DISTANCE, "mae": MAX_RGB_MAE, "aspect": ASPECT_TOLERANCE, "method": 1}:
        raise RuntimeError("Пороги или manifest изменились. Повторите поиск пар перед удалением.")
    cleanup = eda.remove_similar_images(DATA, OUTPUT, pairs, manifest_hash=pair_params["manifest"],
                                        hamming=MAX_DHASH_DISTANCE, mae=MAX_RGB_MAE,
                                        aspect_tolerance=ASPECT_TOLERANCE)
else:
    cleanup = {"removed": 0, "note": "Удаление отключено параметрами."}
print(json.dumps(cleanup, ensure_ascii=False, indent=2))
records, images, objects, names = eda.load_export(OUTPUT)
print("Изображений после проверки:", len(images))

## 5. Распределение классов во всём датасете и по выборкам

Считаем отдельно **объекты** и **изображения с классом**: несколько объектов одного класса на фото увеличивают первый счётчик, а второй — только на единицу. Общие частоты относятся к размеченной части датасета. Для test без меток выводим `NaN`/«неизвестно», а не нули.

В переменных `distribution` и `class_table` доступны все 417 классов, включая отсутствующие в отдельной размеченной выборке.

In [ ]:
distribution = eda.class_distribution(records, names)
object_counts = distribution.pivot(index=["class_id", "class"], columns="split", values="objects")
image_counts = distribution.pivot(index=["class_id", "class"], columns="split", values="images")
class_table = object_counts.add_prefix("objects_").join(image_counts.add_prefix("images_")).reset_index()
display(class_table.sort_values("objects_all", ascending=False))
coverage = []
for split in ["all", "train", "val", "test"]:
    part = distribution[distribution.split == split]
    known = part.objects.notna().any()
    coverage.append({"split": split, "objects": int(part.objects.sum()) if known else None,
                     "classes_present": int((part.objects > 0).sum()) if known else None,
                     "classes_missing": int((part.objects == 0).sum()) if known else None})
display(pd.DataFrame(coverage))
missing_valid = class_table[class_table.objects_val == 0][["class_id", "class", "objects_train"]]
print("Нет примеров в valid:", len(missing_valid)); display(missing_valid)
positive = class_table.loc[class_table.objects_all > 0, "objects_all"]
print("Отношение самого частого класса к самому редкому:", round(positive.max() / positive.min(), 2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
for axis, frame, title in [(axes[0], class_table.nlargest(25, "objects_all"), "25 самых частых классов"),
                           (axes[1], class_table.nsmallest(25, "objects_all"), "25 самых редких классов")]:
    frame = frame.sort_values("objects_all")
    axis.barh(frame["class"], frame.objects_all)
    axis.set(title=title, xlabel="Число объектов"); axis.grid(axis="x", alpha=0.2)
fig.tight_layout(); plt.show(); plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(19, 8))
for axis, split, title in zip(axes, ["train", "val", "test"], ["Train", "Valid", "Test"]):
    part = distribution[distribution.split == split]
    if part.objects.notna().any():
        top = part.nlargest(20, "objects").sort_values("objects")
        axis.barh(top["class"], top.objects); axis.set_xlabel("Число объектов")
    else:
        count = int(images.split.isin(["test", "fpb_test_unlabeled"]).sum())
        axis.text(0.5, 0.5, f"{count:,} изображений\nОткрытая разметка отсутствует\nЧастоты классов неизвестны", ha="center", va="center", transform=axis.transAxes)
        axis.set_xticks([]); axis.set_yticks([])
    axis.set_title(title)
fig.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
def show_split_object_shares(rows, title):
    table = ds.split_summary(rows)
    display(table)
    known = table.loc[["train", "val", "test"]]
    shares = known.object_share.fillna(0) * 100
    fig, axis = plt.subplots(figsize=(9, 4))
    bars = axis.bar(["train", "valid", "test (с метками)"], shares, color=["#4c78a8", "#f58518", "#54a24b"])
    for bar, (_, row) in zip(bars, known.iterrows()):
        axis.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                  f"{100*row.object_share:.2f}%\n{int(row.objects):,} объектов", ha="center", fontsize=10)
    axis.set(title=title, ylabel="Доля от всех размеченных объектов, %", ylim=(0, max(110, shares.max()+10)))
    axis.grid(axis="y", alpha=0.2)
    hidden = int(table.loc["fpb_test_unlabeled", "images"])
    fig.text(0.5, 0.01, f"Test без разметки: {hidden:,} изображений — число объектов неизвестно, в доли не включён.", ha="center", fontsize=9)
    fig.tight_layout(rect=(0, 0.06, 1, 1)); plt.show(); plt.close(fig)
    return table

before_class_table = class_table.copy()
before_split_table = show_split_object_shares(records, "Доли объектов по выборкам до нового разбиения")


## 6. Новое разбиение размеченной части: 80% / 15% / 5%

Доли задаются **по фотографиям**, а не по отдельным рамкам. Из всех размеченных изображений случайно выбираются 5% для test с фиксированным `SEED`; стратификация test не выполняется. Из оставшихся изображений формируются train и valid с точными размерами 80% и 15% исходной размеченной части.

Балансировка train/valid минимизирует взвешенное отклонение числа объектов каждого класса от целевой доли: изображения обмениваются между выборками, их количество остаётся фиксированным. Все объекты фотографии остаются вместе. Поэтому сравниваются **относительные частоты классов**, а не равное абсолютное число объектов. Полное равенство частот при многоклассовых фотографиях и целочисленных количествах не гарантируется; фактические отклонения выводятся ниже.

`fpb_test_unlabeled` не участвует в случайном выборе, балансировке и расчёте долей. Его записи, фотографии и отсутствие меток сохраняются. Поля `source_id` и `source_split` отражают происхождение, а поле `split` — новое разбиение.

При `RESPLIT_DATASET = True` размеченные изображения и метки перемещаются внутри текущей сборки `processed/`. Обновляются manifest, train/val/test.txt, data.yaml с размеченным test, FPB eval и контрольная сумма. После проверки результат фиксируется; при ошибке файлы и metadata восстанавливаются. Повторный запуск с теми же долями, seed и составом сборки использует готовое разбиение (`REUSED`), без повторного переноса. Исходники в `data/raw/` не изменяются.

In [ ]:
hidden_before = {r["source"]+":"+r["source_id"]: r for r in records if r["objects"] is None}
if RESPLIT_DATASET:
    split_result = ds.resplit_export(DATA, OUTPUT, fractions=SPLIT_FRACTIONS, seed=SEED)
else:
    split_result = {"status": "DISABLED"}
print(json.dumps(split_result, ensure_ascii=False, indent=2))
records, images, objects, names = eda.load_export(OUTPUT)
hidden_after = {r["source"]+":"+r["source_id"]: r for r in records if r["objects"] is None}
assert hidden_before == hidden_after, "Неразмеченный test должен остаться неизменным."
print("Test без разметки сохранён:", len(hidden_after), "изображений")


## 7. Распределения после преобразования выборок

Повторно рассчитываем частоты всех классов и доли объектов, отдельно показываем размеченный test. Соотношение фотографий должно быть 80/15/5 с учётом округления; соотношение объектов может немного отличаться, поскольку число объектов на фото различается. В сравнении train/valid нормируем счётчики на общее число объектов соответствующей выборки.

In [ ]:
distribution = eda.class_distribution(records, names)
object_counts = distribution.pivot(index=["class_id", "class"], columns="split", values="objects")
image_counts = distribution.pivot(index=["class_id", "class"], columns="split", values="images")
class_table = object_counts.add_prefix("objects_").join(image_counts.add_prefix("images_")).reset_index()
display(class_table.sort_values("objects_all", ascending=False))
coverage = []
for split in ["all", "train", "val", "test"]:
    part = distribution[distribution.split == split]
    known = part.objects.notna().any()
    coverage.append({"split": split, "objects": int(part.objects.sum()) if known else None,
                     "classes_present": int((part.objects > 0).sum()) if known else None,
                     "classes_missing": int((part.objects == 0).sum()) if known else None})
display(pd.DataFrame(coverage))
missing_valid = class_table[class_table.objects_val == 0][["class_id", "class", "objects_train"]]
print("Нет примеров в valid:", len(missing_valid)); display(missing_valid)
positive = class_table.loc[class_table.objects_all > 0, "objects_all"]
print("Отношение самого частого класса к самому редкому:", round(positive.max() / positive.min(), 2))
after_class_table = class_table.copy()
after_distribution = distribution.copy()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
for axis, frame, title in [(axes[0], class_table.nlargest(25, "objects_all"), "25 самых частых классов"),
                           (axes[1], class_table.nsmallest(25, "objects_all"), "25 самых редких классов")]:
    frame = frame.sort_values("objects_all")
    axis.barh(frame["class"], frame.objects_all)
    axis.set(title=title, xlabel="Число объектов"); axis.grid(axis="x", alpha=0.2)
fig.tight_layout(); plt.show(); plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(19, 8))
for axis, split, title in zip(axes, ["train", "val", "test"], ["Train", "Valid", "Test"]):
    part = distribution[distribution.split == split]
    if part.objects.notna().any():
        top = part.nlargest(20, "objects").sort_values("objects")
        axis.barh(top["class"], top.objects); axis.set_xlabel("Число объектов")
    else:
        count = int(images.split.isin(["test", "fpb_test_unlabeled"]).sum())
        axis.text(0.5, 0.5, f"{count:,} изображений\nОткрытая разметка отсутствует\nЧастоты классов неизвестны", ha="center", va="center", transform=axis.transAxes)
        axis.set_xticks([]); axis.set_yticks([])
    axis.set_title(title)
fig.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
after_split_table = show_split_object_shares(records, "Доли объектов после разбиения 80/15/5")
balance = class_table[["class_id", "class", "objects_train", "objects_val"]].copy()
balance["train_pct"] = balance.objects_train / max(balance.objects_train.sum(), 1) * 100
balance["valid_pct"] = balance.objects_val / max(balance.objects_val.sum(), 1) * 100
balance["difference_pp"] = (balance.train_pct-balance.valid_pct).abs()
print("Максимальное отличие частот train/valid, процентных пунктов:", round(balance.difference_pp.max(), 6))
print("Среднее отличие, процентных пунктов:", round(balance.difference_pp.mean(), 6))
display(balance.nlargest(20, "difference_pp"))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(balance.train_pct, balance.valid_pct, s=12, alpha=0.6)
limit = max(balance.train_pct.max(), balance.valid_pct.max())
axes[0].plot([0, limit], [0, limit], "--", color="gray")
axes[0].set(xlabel="Частота класса в train, %", ylabel="Частота класса в valid, %", title="Все классы: сравнение распределений")
worst = balance.nlargest(15, "difference_pp").sort_values("difference_pp")
axes[1].barh(worst["class"], worst.difference_pp)
axes[1].set(title="Наибольшие отклонения", xlabel="Абсолютная разница, процентных пунктов")
fig.tight_layout(); plt.show(); plt.close(fig)